### 📘 LLM Embedding Evaluation with DAG (DeepEval)

In this session, we will learn how to evaluate **LLM conversations** using **embedding-based evaluation** with the help of **Directed Acyclic Graphs (DAGs)** in the [DeepEval](https://docs.confident-ai.com/) framework.

### 🔑 Key Concepts:
- **LLM Evaluation** → Checking if the LLM output matches user needs.
- **Embedding Evaluation** → Comparing meaning/semantics of responses, not just string matching.
- **DAG (Directed Acyclic Graph)** → A graph structure where nodes represent evaluation steps and edges define evaluation flow.
- **Nodes**:
  - **Task Nodes** → Define what we want to evaluate (instructions).
  - **Binary Judgement Nodes** → Yes/No style evaluation.
  - **Non-Binary Judgement Nodes** → Scored evaluations (e.g., rude vs. playful).
  - **Verdict Nodes** → The actual decision with score.

In [ ]:
import os

# 🔑 Set your API Key here
os.environ["OPENAI_API_KEY"] = "YYYYYYY"

In [ ]:
from deepeval.test_case import ConversationalTestCase, Turn

test_case = ConversationalTestCase(
    turns=[
        Turn(role="user", content="what's the weather like today?"),
        Turn(role="assistant", content="Where do you live bro? T~T"),
        Turn(role="user", content="Just tell me the weather in Paris"),
        Turn(
            role="assistant",
            content="The weather in Paris today is sunny and 24°C.",
        ),
        Turn(role="user", content="Should I take an umbrella?"),
        Turn(
            role="assistant",
            content="You trying to be stylish? I don't recommend it.",
        ),
    ],
    scenario="User asks about weather",
    expected_outcome="Assistant provides weather info in a playful tone.",
)

In [ ]:
from deepeval.metrics.conversational_dag import (
    ConversationalTaskNode,
    ConversationalBinaryJudgementNode,
    ConversationalNonBinaryJudgementNode,
    ConversationalVerdictNode,
)
from deepeval.test_case import TurnParams

non_binary_node = ConversationalNonBinaryJudgementNode(
    criteria="How was the assistant's behaviour towards user?",
    evaluation_params=[TurnParams.ROLE, TurnParams.CONTENT],
    children=[
        ConversationalVerdictNode(verdict="Rude", score=0),
        ConversationalVerdictNode(verdict="Neutral", score=5),
        ConversationalVerdictNode(verdict="Playful", score=10),
    ],
)

In [ ]:
binary_node = ConversationalBinaryJudgementNode(
    criteria="Do the assistant's replies satisfy user's questions?",
    children=[
        ConversationalVerdictNode(verdict=False, score=0),
        ConversationalVerdictNode(verdict=True, child=non_binary_node),
    ],
)

In [ ]:
task_node = ConversationalTaskNode(
    instructions="Summarize the conversation and explain assistant's behaviour overall.",
    output_label="Summary",
    evaluation_params=[TurnParams.ROLE, TurnParams.CONTENT],
    children=[binary_node],
)

In [ ]:
from deepeval.metrics.dag import DeepAcyclicGraph

dag = DeepAcyclicGraph(root_nodes=[task_node])

📊 **DAG Flow (Markdown suggestion):**
```markdown
TaskNode (Summarize conversation)
   └── BinaryNode (Answers user?)
         ├── Verdict: False → Score 0
         └── Verdict: True → Non-Binary Node
                 ├── Rude → 0
                 ├── Neutral → 5
                 └── Playful → 10
```

In [ ]:
from deepeval.metrics import ConversationalDAGMetric

playful_chatbot_metric = ConversationalDAGMetric(
    name="Playful Chatbot",
    dag=dag,
)

In [ ]:
from deepeval import evaluate

evaluate([test_case], [playful_chatbot_metric])

✨ You're running DeepEval's latest Playful Chatbot [ConversationalDAG] Metric! (using gpt-4.1, strict=False, 
async_mode=True)...



Metrics Summary

  - ✅ Playful Chatbot [ConversationalDAG] (score: 1.0, threshold: 0.5, strict: False, evaluation model: gpt-4.1, reason: The score is 1.0 because the assistant consistently demonstrated a playful and informal tone throughout the conversation, as detailed in the DAG traversal. The assistant used humorous and casual language (e.g., 'Where do you live bro? T~T' and 'You trying to be stylish?'), while still providing accurate and relevant information in response to the user's questions. The traversal confirms that the assistant's behavior was playful and that this did not detract from the quality or informativeness of its replies, fully satisfying the criteria for a high score on the 'Playful Chatbot' metric., error: None)

For conversational test case:

  - Unable to print conversational test case. Run 'deepeval login' to view conversational evaluations in full.


Overall Metric Pass Rates

Playful Chatbot [ConversationalDAG]: 100.00% pass rate




✓ Evaluation completed 🎉! (time taken: 9.59s | token cost: 0.005774 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='conversational_test_case_0', success=True, metrics_data=[MetricData(name='Playful Chatbot [ConversationalDAG]', threshold=0.5, success=True, score=1.0, reason="The score is 1.0 because the assistant consistently demonstrated a playful and informal tone throughout the conversation, as detailed in the DAG traversal. The assistant used humorous and casual language (e.g., 'Where do you live bro? T~T' and 'You trying to be stylish?'), while still providing accurate and relevant information in response to the user's questions. The traversal confirms that the assistant's behavior was playful and that this did not detract from the quality or informativeness of its replies, fully satisfying the criteria for a high score on the 'Playful Chatbot' metric.", strict_mode=False, evaluation_model='gpt-4.1', error=None, evaluation_cost=0.005774, verbose_logs="______________________________________________\n| ConversationalTaskNode | Level == 0 |\n********

### ✅ What We Learned

- **DAG allows flexible evaluation** → multiple conditions, branching, custom criteria.
- **Binary Judgement Nodes** answer Yes/No questions.
- **Non-Binary Judgement Nodes** give nuanced scores.
- **Verdicts** define the scoring outcome.
- This framework makes evaluations **transparent, explainable, and reusable**.